# Bangla News Framing: target-aware two-stage training

This notebook builds an interpretable system for comparing how Bangla news articles about the same event frame the government. It predicts sentence–aspect relevance first, predicts government-directed stance only for relevant pairs, aggregates probability-based article profiles, and compares same-event articles.

**Fixed design:** `csebuetnlp/banglabert`; six fixed political-news aspects; labels `NR`, `N`, `GF`, and `GC`; event-grouped splits; and no source, URL, article label, or identifier features in the sentence models. The annotations are AI-assisted and must not be presented as a fully human-verified gold standard.

## 1. Environment setup and reproducibility

Select a GPU runtime in Colab before running this section. The package command is rerunnable: packages already available in the runtime are retained when they satisfy the requirement.

In [ ]:
%pip install -q torch transformers datasets accelerate evaluate scikit-learn pandas numpy matplotlib seaborn pyarrow joblib

In [ ]:
import importlib.metadata as importlib_metadata
import json
import os
import platform
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from transformers import set_seed as set_transformers_seed

PACKAGE_NAMES = [
    "torch", "transformers", "datasets", "accelerate", "evaluate",
    "scikit-learn", "pandas", "numpy", "matplotlib", "seaborn",
    "pyarrow", "joblib",
]
PACKAGE_VERSIONS = {
    name: importlib_metadata.version(name) for name in PACKAGE_NAMES
}

print(f"Python: {platform.python_version()}")
for name, version in PACKAGE_VERSIONS.items():
    print(f"{name}: {version}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime reported by PyTorch: {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. In Colab, choose Runtime > Change runtime "
        "type > GPU, reconnect, and rerun the notebook from the beginning."
    )

## 2. Configuration, Drive paths, and deterministic seeds

All paths and initial hyperparameters are defined once here. Colab displays the folder as **My Drive**, while its mounted filesystem path is `/content/drive/MyDrive`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

SEED = 42
PROJECT_ROOT = Path("/content/drive/MyDrive/bangla-news-framing")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PATH = RAW_DATA_DIR / "banglabias_sentence_annotations_completed.csv"
OUTPUT_DIR = PROJECT_ROOT / "artifacts"

CONFIG = {
    "model_name": "csebuetnlp/banglabert",
    "seed": SEED,
    "data_path": str(DATA_PATH),
    "output_dir": str(OUTPUT_DIR),
    "max_length": 128,
    "learning_rate": 2e-5,
    "train_batch_size": 16,
    "eval_batch_size": 32,
    "max_epochs": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "early_stopping_patience": 2,
}

def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    set_transformers_seed(seed)

set_all_seeds(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Place the completed CSV in "
        "My Drive/bangla-news-framing/data/raw and rerun this cell."
    )

ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0),
    "pytorch_cuda": torch.version.cuda,
    "packages": PACKAGE_VERSIONS,
}

with (OUTPUT_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump(CONFIG, file, ensure_ascii=False, indent=2)
with (OUTPUT_DIR / "environment.json").open("w", encoding="utf-8") as file:
    json.dump(ENVIRONMENT, file, ensure_ascii=False, indent=2)

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print(f"Artifacts will be saved to: {OUTPUT_DIR}")

## 3. Load and strictly validate the annotation CSV

The source CSV remains unchanged. Empty strings are preserved during loading, and assertions stop execution if the file differs from the authoritative schema or expected dataset statistics.

In [ ]:
EXPECTED_COLUMNS = [
    "article_id",
    "event_name",
    "news_headline",
    "news_source",
    "publication_date",
    "source_link",
    "article_stance",
    "sentence_id",
    "sentence_index",
    "sentence_section",
    "sentence_text",
    "a1_policy_administration",
    "a2_accountability_justice",
    "a3_security_civil_rights",
    "a4_democracy_mobilization",
    "a5_economy_development",
    "a6_welfare_social_environment",
    "stance_holder",
    "is_quote",
]
ASPECT_COLUMNS = EXPECTED_COLUMNS[11:17]
ALLOWED_ASPECT_LABELS = {"NR", "N", "GF", "GC"}
ALLOWED_ARTICLE_LABELS = {"Govt critique", "Neutral", "Govt leaning"}
ALLOWED_SECTIONS = {"headline", "lead", "body"}
ALLOWED_STANCE_HOLDERS = {
    "journalist", "government", "opposition", "law_enforcement",
    "court", "expert", "activist", "citizen", "victim_or_family",
    "organization", "unclear",
}

df = pd.read_csv(DATA_PATH, encoding="utf-8", keep_default_na=False)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
display(pd.DataFrame({"dtype": df.dtypes.astype(str)}))

In [ ]:
def blank_count(frame: pd.DataFrame) -> int:
    return int(frame.astype(str).apply(lambda column: column.str.strip().eq("").sum()).sum())

assert df.columns.tolist() == EXPECTED_COLUMNS, (
    "Unexpected schema or column order.\n"
    f"Expected: {EXPECTED_COLUMNS}\nFound: {df.columns.tolist()}"
)
assert df.shape == (5308, 19), f"Expected shape (5308, 19), found {df.shape}"
assert df["article_id"].nunique() == 118, "Expected 118 unique articles"
assert df["event_name"].nunique() == 26, "Expected 26 unique events"
assert df["sentence_id"].duplicated().sum() == 0, "Duplicate sentence_id values found"
assert df["sentence_text"].str.strip().ne("").all(), "Blank sentence_text found"
assert blank_count(df) == 0, "Unexpected blank cells found in the completed dataset"
assert set(df["article_stance"].unique()) == ALLOWED_ARTICLE_LABELS
assert set(df["sentence_section"].unique()) == ALLOWED_SECTIONS
assert set(df["stance_holder"].unique()) == ALLOWED_STANCE_HOLDERS
assert set(df["is_quote"].unique()) == {0, 1}
assert df.groupby("article_id")["event_name"].nunique().max() == 1
assert df.groupby("article_id")["article_stance"].nunique().max() == 1

for column in ASPECT_COLUMNS:
    found_labels = set(df[column].unique())
    assert found_labels == ALLOWED_ASPECT_LABELS, (
        f"{column}: expected {sorted(ALLOWED_ASPECT_LABELS)}, "
        f"found {sorted(found_labels)}"
    )

article_six_dates = set(
    df.loc[df["article_id"].astype(str).eq("6"), "publication_date"].astype(str)
)
assert article_six_dates == {"2023-09-23"}, (
    f"Article 6 must have corrected date 2023-09-23; found {article_six_dates}"
)

article_rows = df.drop_duplicates("article_id")
audit_summary = {
    "rows": len(df),
    "columns": df.shape[1],
    "articles": int(df["article_id"].nunique()),
    "events": int(df["event_name"].nunique()),
    "duplicate_sentence_ids": int(df["sentence_id"].duplicated().sum()),
    "blank_sentence_texts": int(df["sentence_text"].str.strip().eq("").sum()),
    "blank_cells": blank_count(df),
}
display(pd.Series(audit_summary, name="value").to_frame())
print("All schema and integrity assertions passed.")

## 4. Initial data audit and class imbalance

Article labels are counted once per article. Aspect labels are counted over sentence rows. Accuracy will not be treated as the primary metric because `NR` dominates relevance and `N` dominates relevant-pair stance.

In [ ]:
article_label_counts = (
    article_rows["article_stance"]
    .value_counts()
    .reindex(["Govt leaning", "Neutral", "Govt critique"])
)
section_counts = (
    df["sentence_section"]
    .value_counts()
    .reindex(["headline", "lead", "body"])
)
aspect_label_counts = pd.DataFrame({
    column: df[column].value_counts() for column in ASPECT_COLUMNS
}).T.reindex(columns=["NR", "N", "GF", "GC"]).fillna(0).astype(int)
aspect_label_percentages = aspect_label_counts.div(
    aspect_label_counts.sum(axis=1), axis=0
).mul(100).round(2)

assert article_label_counts.to_dict() == {
    "Govt leaning": 33, "Neutral": 45, "Govt critique": 40
}
assert section_counts.to_dict() == {"headline": 118, "lead": 118, "body": 5072}
assert aspect_label_counts.sum().to_dict() == {
    "NR": 24630, "N": 6882, "GF": 108, "GC": 228
}

print("Article labels (one row per article)")
display(article_label_counts.rename("count").to_frame())
print("Sentence sections")
display(section_counts.rename("count").to_frame())
print("Aspect-label counts")
display(aspect_label_counts)
print("Aspect-label percentages")
display(aspect_label_percentages)

In [ ]:
sentence_lengths = df["sentence_text"].str.len()
length_summary = sentence_lengths.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).round(2)
print("Sentence length in Unicode characters")
display(length_summary.rename("characters").to_frame())

for section in ["headline", "lead", "body"]:
    print(f"Examples — {section}")
    display(
        df.loc[df["sentence_section"].eq(section), ["sentence_id", "sentence_text"]]
        .head(3)
        .style.set_properties(subset=["sentence_text"], **{"text-align": "left"})
    )

In [ ]:
plot_data = (
    aspect_label_counts
    .rename_axis("aspect_column")
    .reset_index()
    .melt(id_vars="aspect_column", var_name="label", value_name="count")
)
plt.figure(figsize=(13, 5))
sns.barplot(
    data=plot_data, x="aspect_column", y="count", hue="label",
    hue_order=["NR", "N", "GF", "GC"],
)
plt.title("Sentence-level labels by aspect")
plt.xlabel("Aspect")
plt.ylabel("Sentence count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

audit_report = {
    **audit_summary,
    "article_label_counts": {key: int(value) for key, value in article_label_counts.items()},
    "sentence_section_counts": {key: int(value) for key, value in section_counts.items()},
    "aspect_label_counts": {
        aspect: {label: int(value) for label, value in counts.items()}
        for aspect, counts in aspect_label_counts.to_dict(orient="index").items()
    },
    "sentence_length_characters": {
        key: float(value) for key, value in length_summary.items()
    },
}
with (OUTPUT_DIR / "data_audit.json").open("w", encoding="utf-8") as file:
    json.dump(audit_report, file, ensure_ascii=False, indent=2)

print(f"Saved configuration, environment, and audit metadata to {OUTPUT_DIR}")

### Checkpoint

At this point the runtime, paths, dataset schema, integrity, and documented distributions have been verified. The next increment will add conservative Bangla text cleaning with before/after diagnostics, the fixed aspect mapping, and wide-to-long conversion.